## CogVideoX Image-to-Video


#### Install the necessary requirements

In [ ]:
!pip install diffusers transformers hf_transfer

In [ ]:
# !pip install git+https://github.com/huggingface/accelerate
!pip install accelerate==0.33.0

#### Import required libraries

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import torch
from diffusers import AutoencoderKLCogVideoX, CogVideoXImageToVideoPipeline, CogVideoXTransformer3DModel
from diffusers.utils import export_to_video, load_image
from transformers import T5EncoderModel

#### Helper Functions

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Function to display video
def display_video(video_path):
    # Read video file and encode in base64
    video = open(video_path, "rb").read()
    video_encoded = b64encode(video).decode('ascii')

    # Create HTML5 video tag with encoded video
    video_tag = f'''
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_encoded}" type="video/mp4">
    </video>
    '''
    return HTML(video_tag)

#### Load models and create pipeline



In [ ]:
model_id = "THUDM/CogVideoX-5b-I2V"

In [ ]:
transformer = CogVideoXTransformer3DModel.from_pretrained(model_id, subfolder="transformer", torch_dtype=torch.float16)
text_encoder = T5EncoderModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.float16)
vae = AutoencoderKLCogVideoX.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Create pipeline and run inference
pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    transformer=transformer,
    vae=vae,
    torch_dtype=torch.float16,
)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

#### Enable memory optimizations


In [ ]:
pipe.enable_sequential_cpu_offload()

#### Generate!

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving image.jpeg to image (1).jpeg


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image = Image.open("image.jpeg")

# Display the image using matplotlib
plt.imshow(image)
plt.axis('off')  # Hide the axes for a cleaner look
plt.show()

In [ ]:
prompt = "The person is holding a flask in his right hand and gently swirling it, while his left hand stabilizes the tube attached to a lab stand."

In [ ]:
video = pipe(image=image, prompt=prompt, guidance_scale=6, use_dynamic_cfg=True, num_inference_steps=50).frames[0]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
export_to_video(video, "output.mp4", fps=8)

'output.mp4'

In [ ]:
display_video("output.mp4")

720 x 480

https://huggingface.co/spaces/THUDM/CogVideoX-5B-Space